In [40]:
import numpy as np
import copy

from utils.import_puzzle_input import load_input
from utils.import_puzzle_input import matrix_puzzle_input

# Load the autoreload extension
%load_ext autoreload

# Set autoreload to automatically reload modules before executing code
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
puzzle_input = load_input(2023, 4)

loaded puzzle input


In [42]:
puzzle_input_split = puzzle_input.split("\n")
puzzle_input_split

['Card   1: 34 50 18 44 19 35 47 62 65 26 | 63  6 27 15 60  9 98  3 61 89 31 43 80 37 54 49 92 55  8  7 10 16 52 33 45',
 'Card   2: 90 12 98 56 22 99 73 46  1 28 | 52 77 32  8 81 41 53 22 28 46 48 27 98  1 94 12 99 72 84 90 92 73 24 63 56',
 'Card   3: 48 10 39 87 23 78 49 40 55  8 | 48 80 78 87 39 24 27 19 41 73 30 52 10  2 67 40 88 53 59 84 55 49  5 33 82',
 'Card   4: 21 45 91 26 64 51 42 84 11 94 | 55 56 36 65 84  2 68 44 52 58 86  6 33  7 97 40 30 14 39 80 82 57 79  1 10',
 'Card   5: 33  6 67 89 64 31 85 11  2 15 |  6 70 29 89 12 11 64 80  7 82 46 16 33 68 48 72 31  2 99 15 67 57  4 49 85',
 'Card   6: 51 20 11 66 38 39 69 48 25 74 | 39 74  3 86 19 25 21 55  2 38 46 60 66 82 51 11 98 88  8 48 49 94 20 69 72',
 'Card   7:  4 50 82 51 52 77 12 11 57 42 | 56 11 73 69 42 82 32 77 52 98 12 51 36 94 46  4 50 39 85 90 93 70 18 71 57',
 'Card   8: 96 31 27 93  7  8  6 23 15 72 | 55 79 86  4  6 35 12 27 95 29 73 81 87 43  7 13 62 15 72 71 58 48 63 94 89',
 'Card   9: 16 90 79 29 93 31 40

# Problem Setup

--- Day 4: Scratchcards ---
The gondola takes you up. Strangely, though, the ground doesn't seem to be coming with you; you're not climbing a mountain. As the circle of Snow Island recedes below you, an entire new landmass suddenly appears above you! The gondola carries you to the surface of the new island and lurches into the station.  
  
As you exit the gondola, the first thing you notice is that the air here is much warmer than it was on Snow Island. It's also quite humid. Is this where the water source is?  
  
The next thing you notice is an Elf sitting on the floor across the station in what seems to be a pile of colorful square cards.  
  
"Oh! Hello!" The Elf excitedly runs over to you. "How may I be of service?" You ask about water sources.  
  
"I'm not sure; I just operate the gondola lift. That does sound like something we'd have, though - this is Island Island, after all! I bet the gardener would know. He's on a different island, though - er, the small kind surrounded by water, not the floating kind. We really need to come up with a better naming scheme. Tell you what: if you can help me with something quick, I'll let you borrow my boat and you can go visit the gardener. I got all these scratchcards as a gift, but I can't figure out what I've won."  
  
The Elf leads you over to the pile of colorful cards. There, you discover dozens of scratchcards, all with their opaque covering already scratched off. Picking one up, it looks like each card has two lists of numbers separated by a vertical bar (|): a list of winning numbers and then a list of numbers you have. You organize the information into a table (your puzzle input).  
  
As far as the Elf has been able to figure out, you have to figure out which of the numbers you have appear in the list of winning numbers. The first match makes the card worth one point and each match after the first doubles the point value of that card.  
  
For example:  
  
Card 1: 41 48 83 86 17 | 83 86  6 31 17  9 48 53  
Card 2: 13 32 20 16 61 | 61 30 68 82 17 32 24 19  
Card 3:  1 21 53 59 44 | 69 82 63 72 16 21 14  1  
Card 4: 41 92 73 84 69 | 59 84 76 51 58  5 54 83  
Card 5: 87 83 26 28 32 | 88 30 70 12 93 22 82 36  
Card 6: 31 18 13 56 72 | 74 77 10 23 35 67 36 11  
In the above example, card 1 has five winning numbers (41, 48, 83, 86, and 17) and eight numbers you have (83, 86, 6, 31, 17, 9, 48, and 53). Of the numbers you have, four of them (48, 83, 17, and 86) are winning numbers! That means card 1 is worth 8 points (1 for the first match, then doubled three times for each of the three matches after the first).  
  
Card 2 has two winning numbers (32 and 61), so it is worth 2 points.  
Card 3 has two winning numbers (1 and 21), so it is worth 2 points.  
Card 4 has one winning number (84), so it is worth 1 point.  
Card 5 has no winning numbers, so it is worth no points.  
Card 6 has no winning numbers, so it is worth no points.  
So, in this example, the Elf's pile of scratchcards is worth 13 points.  
  
Take a seat in the large pile of colorful cards. How many points are they worth in total?  

# General thoughts

ASSUMPTIONS:  
1. Neither the winning numbers or the scrathcard numbers will have the same number appear twice.

# Solutions

## Solution 1:

### Explanation of method

1. For each scratch-card:  
    1.1. Extract the numbers on each side of `|`  
    1.2. Store the winning numbers in a set `winning_numbers`, store the scratch-card numbers in a set `scratchcard_numbers`.  
    1.3. Take the intersection of `winning_numbers` and `scratchcard_numbers`, measure its length as `len_matching_numbers`  
    1.4. The number of points for that scratch card will be `2**len_matching_numbers`  
2. Sum the number of points for each of the games

### Implementation

In [43]:
puzzle_input_refined = [line.split(": ")[1] for line in puzzle_input_split]
puzzle_input_refined

['34 50 18 44 19 35 47 62 65 26 | 63  6 27 15 60  9 98  3 61 89 31 43 80 37 54 49 92 55  8  7 10 16 52 33 45',
 '90 12 98 56 22 99 73 46  1 28 | 52 77 32  8 81 41 53 22 28 46 48 27 98  1 94 12 99 72 84 90 92 73 24 63 56',
 '48 10 39 87 23 78 49 40 55  8 | 48 80 78 87 39 24 27 19 41 73 30 52 10  2 67 40 88 53 59 84 55 49  5 33 82',
 '21 45 91 26 64 51 42 84 11 94 | 55 56 36 65 84  2 68 44 52 58 86  6 33  7 97 40 30 14 39 80 82 57 79  1 10',
 '33  6 67 89 64 31 85 11  2 15 |  6 70 29 89 12 11 64 80  7 82 46 16 33 68 48 72 31  2 99 15 67 57  4 49 85',
 '51 20 11 66 38 39 69 48 25 74 | 39 74  3 86 19 25 21 55  2 38 46 60 66 82 51 11 98 88  8 48 49 94 20 69 72',
 ' 4 50 82 51 52 77 12 11 57 42 | 56 11 73 69 42 82 32 77 52 98 12 51 36 94 46  4 50 39 85 90 93 70 18 71 57',
 '96 31 27 93  7  8  6 23 15 72 | 55 79 86  4  6 35 12 27 95 29 73 81 87 43  7 13 62 15 72 71 58 48 63 94 89',
 '16 90 79 29 93 31 40 24 82 88 | 86 16 73 20 22 93 83 39 36 90 79 72 40 29 35 97 88 12  8 24 31 82 21 59 95',
 

In [44]:
puzzle_input_refined = [line.replace("  ", " ") for line in puzzle_input_refined]

In [45]:
puzzle_input_refined

['34 50 18 44 19 35 47 62 65 26 | 63 6 27 15 60 9 98 3 61 89 31 43 80 37 54 49 92 55 8 7 10 16 52 33 45',
 '90 12 98 56 22 99 73 46 1 28 | 52 77 32 8 81 41 53 22 28 46 48 27 98 1 94 12 99 72 84 90 92 73 24 63 56',
 '48 10 39 87 23 78 49 40 55 8 | 48 80 78 87 39 24 27 19 41 73 30 52 10 2 67 40 88 53 59 84 55 49 5 33 82',
 '21 45 91 26 64 51 42 84 11 94 | 55 56 36 65 84 2 68 44 52 58 86 6 33 7 97 40 30 14 39 80 82 57 79 1 10',
 '33 6 67 89 64 31 85 11 2 15 | 6 70 29 89 12 11 64 80 7 82 46 16 33 68 48 72 31 2 99 15 67 57 4 49 85',
 '51 20 11 66 38 39 69 48 25 74 | 39 74 3 86 19 25 21 55 2 38 46 60 66 82 51 11 98 88 8 48 49 94 20 69 72',
 ' 4 50 82 51 52 77 12 11 57 42 | 56 11 73 69 42 82 32 77 52 98 12 51 36 94 46 4 50 39 85 90 93 70 18 71 57',
 '96 31 27 93 7 8 6 23 15 72 | 55 79 86 4 6 35 12 27 95 29 73 81 87 43 7 13 62 15 72 71 58 48 63 94 89',
 '16 90 79 29 93 31 40 24 82 88 | 86 16 73 20 22 93 83 39 36 90 79 72 40 29 35 97 88 12 8 24 31 82 21 59 95',
 '17 36 50 39 96 43 41 38 55 8 | 

In [46]:
puzzle_input_refined = [line[0].replace(" ", "") + line[1:] for line in puzzle_input_refined]

In [47]:
winning_numbers_our_numbers = [line.split(" | ") for line in puzzle_input_refined]
winning_numbers_our_numbers

[['34 50 18 44 19 35 47 62 65 26',
  '63 6 27 15 60 9 98 3 61 89 31 43 80 37 54 49 92 55 8 7 10 16 52 33 45'],
 ['90 12 98 56 22 99 73 46 1 28',
  '52 77 32 8 81 41 53 22 28 46 48 27 98 1 94 12 99 72 84 90 92 73 24 63 56'],
 ['48 10 39 87 23 78 49 40 55 8',
  '48 80 78 87 39 24 27 19 41 73 30 52 10 2 67 40 88 53 59 84 55 49 5 33 82'],
 ['21 45 91 26 64 51 42 84 11 94',
  '55 56 36 65 84 2 68 44 52 58 86 6 33 7 97 40 30 14 39 80 82 57 79 1 10'],
 ['33 6 67 89 64 31 85 11 2 15',
  '6 70 29 89 12 11 64 80 7 82 46 16 33 68 48 72 31 2 99 15 67 57 4 49 85'],
 ['51 20 11 66 38 39 69 48 25 74',
  '39 74 3 86 19 25 21 55 2 38 46 60 66 82 51 11 98 88 8 48 49 94 20 69 72'],
 ['4 50 82 51 52 77 12 11 57 42',
  '56 11 73 69 42 82 32 77 52 98 12 51 36 94 46 4 50 39 85 90 93 70 18 71 57'],
 ['96 31 27 93 7 8 6 23 15 72',
  '55 79 86 4 6 35 12 27 95 29 73 81 87 43 7 13 62 15 72 71 58 48 63 94 89'],
 ['16 90 79 29 93 31 40 24 82 88',
  '86 16 73 20 22 93 83 39 36 90 79 72 40 29 35 97 88 12 8 24 31 82 2

In [48]:
winning_numbers, our_numbers = zip(*[line.split(" | ") for line in puzzle_input_refined])

In [49]:
winning_numbers[0]

'34 50 18 44 19 35 47 62 65 26'

In [50]:
our_numbers[0]

'63 6 27 15 60 9 98 3 61 89 31 43 80 37 54 49 92 55 8 7 10 16 52 33 45'

In [51]:
common_numbers_in_game = [set(winning_numbers[i].split(" ")) & set(our_numbers[i].split(" ")) for i in range(len(our_numbers))]
common_numbers_in_game

[set(),
 {'1', '12', '22', '28', '46', '56', '73', '90', '98', '99'},
 {'10', '39', '40', '48', '49', '55', '78', '87'},
 {'84'},
 {'11', '15', '2', '31', '33', '6', '64', '67', '85', '89'},
 {'11', '20', '25', '38', '39', '48', '51', '66', '69', '74'},
 {'11', '12', '4', '42', '50', '51', '52', '57', '77', '82'},
 {'15', '27', '6', '7', '72'},
 {'16', '24', '29', '31', '40', '79', '82', '88', '90', '93'},
 {'17', '36', '38', '39', '41', '43', '50', '55', '8', '96'},
 {'11', '24', '28', '64', '76', '93', '97'},
 {'24', '25', '39', '54', '55', '89'},
 set(),
 {'22', '26', '3', '47', '66', '77'},
 {'43', '91'},
 {'3', '35', '46', '79', '80', '95'},
 {'50', '52', '89'},
 {'15', '40', '57'},
 {'41'},
 set(),
 {'90'},
 set(),
 {'10', '29', '30', '32', '57', '69', '72', '78', '79', '86'},
 set(),
 {'17', '28', '34', '6', '87', '89'},
 {'14', '2', '23', '37', '42', '53', '59', '62', '8', '82'},
 {'11', '17', '30', '39', '56', '58', '61', '75', '93', '95'},
 {'10', '26', '28', '34', '35', '5',

In [52]:
points = [2**(len(common_numbers)-1) for common_numbers in common_numbers_in_game if common_numbers]
points

[512,
 128,
 1,
 512,
 512,
 512,
 16,
 512,
 512,
 64,
 32,
 32,
 2,
 32,
 4,
 4,
 1,
 1,
 512,
 32,
 512,
 512,
 512,
 64,
 32,
 64,
 1,
 8,
 2,
 8,
 16,
 2,
 1,
 1,
 64,
 512,
 512,
 32,
 256,
 16,
 4,
 8,
 32,
 16,
 8,
 1,
 2,
 512,
 64,
 16,
 32,
 16,
 32,
 64,
 4,
 16,
 4,
 2,
 2,
 1,
 512,
 512,
 512,
 512,
 4,
 512,
 4,
 512,
 512,
 4,
 512,
 512,
 512,
 4,
 32,
 128,
 1,
 1,
 2,
 16,
 1,
 2,
 1,
 1,
 128,
 64,
 512,
 512,
 16,
 16,
 128,
 32,
 256,
 128,
 64,
 4,
 8,
 1,
 4,
 1,
 2,
 1,
 512,
 128,
 512,
 512,
 512,
 2,
 8,
 32,
 32,
 2,
 8,
 1,
 2,
 1,
 1,
 8,
 128,
 512,
 8,
 128,
 8,
 64,
 32,
 1,
 16,
 64,
 1,
 16,
 8,
 16,
 4,
 1,
 2,
 2,
 512,
 512,
 512,
 512,
 2,
 256,
 512,
 512,
 128,
 16,
 32,
 64,
 32,
 64,
 256,
 64,
 2,
 16,
 4,
 8,
 4,
 1,
 1,
 256,
 256,
 128,
 512,
 32,
 32,
 1,
 32,
 16,
 8,
 4,
 1]

In [53]:
np.sum(points)

np.int64(23678)

## Part 2:

Memoization is an approach here. We note that we at some point above lost the connection of the number of common numbers to its original card. This connection is neccesary in part 2, so we might either redo part 1, with this in mind, or create something special for part 2.
We choose to redo part 1 below here:

### redoing part 1 with specific cards:

In [54]:
def step_1(puzzle_input_split: list[str]) -> dict[int, str]:
    # We:
    # - Collapse double space to a single space. 
    # We do this to standardise spacings between numbers.
    # The spacings are uneven as some numbers are double digit,
    # and some single digit. This leaves an extra space before 
    # single digits to keep numbers human-readable in the input.
    # - Remove the 'cards x: ' specification.
    # - Replace double spacings before splitting on ": " to not
    # leave an extra space on those rows where the first number
    # is single digit.
    # - We finally place the puzzle input in a dict with key equal to the card number.
    return  {
        i+1:
            puzzle_input_split[i].replace("  ", " ").split(": ")[1]
            for i in range(len(puzzle_input_split))
    }

In [55]:
puzzle_input_refined = step_1(puzzle_input_split)
puzzle_input_refined

{1: '34 50 18 44 19 35 47 62 65 26 | 63 6 27 15 60 9 98 3 61 89 31 43 80 37 54 49 92 55 8 7 10 16 52 33 45',
 2: '90 12 98 56 22 99 73 46 1 28 | 52 77 32 8 81 41 53 22 28 46 48 27 98 1 94 12 99 72 84 90 92 73 24 63 56',
 3: '48 10 39 87 23 78 49 40 55 8 | 48 80 78 87 39 24 27 19 41 73 30 52 10 2 67 40 88 53 59 84 55 49 5 33 82',
 4: '21 45 91 26 64 51 42 84 11 94 | 55 56 36 65 84 2 68 44 52 58 86 6 33 7 97 40 30 14 39 80 82 57 79 1 10',
 5: '33 6 67 89 64 31 85 11 2 15 | 6 70 29 89 12 11 64 80 7 82 46 16 33 68 48 72 31 2 99 15 67 57 4 49 85',
 6: '51 20 11 66 38 39 69 48 25 74 | 39 74 3 86 19 25 21 55 2 38 46 60 66 82 51 11 98 88 8 48 49 94 20 69 72',
 7: '4 50 82 51 52 77 12 11 57 42 | 56 11 73 69 42 82 32 77 52 98 12 51 36 94 46 4 50 39 85 90 93 70 18 71 57',
 8: '96 31 27 93 7 8 6 23 15 72 | 55 79 86 4 6 35 12 27 95 29 73 81 87 43 7 13 62 15 72 71 58 48 63 94 89',
 9: '16 90 79 29 93 31 40 24 82 88 | 86 16 73 20 22 93 83 39 36 90 79 72 40 29 35 97 88 12 8 24 31 82 21 59 95',
 10: '1

In [56]:
def step_2(result_of_step_1: dict[int, str]) -> dict[int, list[set, set]]:
    return {
    key: [
        set(number_type.split(" "))
        for number_type in value.split(" | ")
    ]
    for key, value in result_of_step_1.items()
}

In [57]:
result_of_step_2 = step_2(step_1(puzzle_input_split))
result_of_step_2

{1: [{'18', '19', '26', '34', '35', '44', '47', '50', '62', '65'},
  {'10',
   '15',
   '16',
   '27',
   '3',
   '31',
   '33',
   '37',
   '43',
   '45',
   '49',
   '52',
   '54',
   '55',
   '6',
   '60',
   '61',
   '63',
   '7',
   '8',
   '80',
   '89',
   '9',
   '92',
   '98'}],
 2: [{'1', '12', '22', '28', '46', '56', '73', '90', '98', '99'},
  {'1',
   '12',
   '22',
   '24',
   '27',
   '28',
   '32',
   '41',
   '46',
   '48',
   '52',
   '53',
   '56',
   '63',
   '72',
   '73',
   '77',
   '8',
   '81',
   '84',
   '90',
   '92',
   '94',
   '98',
   '99'}],
 3: [{'10', '23', '39', '40', '48', '49', '55', '78', '8', '87'},
  {'10',
   '19',
   '2',
   '24',
   '27',
   '30',
   '33',
   '39',
   '40',
   '41',
   '48',
   '49',
   '5',
   '52',
   '53',
   '55',
   '59',
   '67',
   '73',
   '78',
   '80',
   '82',
   '84',
   '87',
   '88'}],
 4: [{'11', '21', '26', '42', '45', '51', '64', '84', '91', '94'},
  {'1',
   '10',
   '14',
   '2',
   '30',
   '33',
   '36',
 

In [58]:
def step_3(result_of_step_2: dict[int, list[set, set]]) -> dict[int, int]:
    return {
    key: 
        len(value[0] & value[1])
    for key, value in result_of_step_2.items()
}

In [59]:
result_of_step_3 = step_3(step_2(step_1(puzzle_input_split)))

In [60]:
# Let's see the first 10
{key: result_of_step_3[key] for key in list(result_of_step_3)[:10]}

{1: 0, 2: 10, 3: 8, 4: 1, 5: 10, 6: 10, 7: 10, 8: 5, 9: 10, 10: 10}

In [61]:
# check that we get same result
np.sum([2**(number-1) for number in result_of_step_3.values() if number]) # Note important 'if number'

np.int64(23678)

### back to part 2

In [62]:
def memo():
    count = 0
    memo_dict = {}
    def inner(input):
        if input in memo_dict:
            return memo_dict[input]
        else:
            result = input  # Replace this with the actual computation
            memo_dict[input] = result
            return result
    return inner

In [63]:
# original 1: only itself - 1
# original 2: copies of 3, 4, 5, 6, 7, 8, 9, 10, 11, 12 - 1 original and copies of ten cards
# original 3: original, 

In [64]:
test_case_copied = """
    Card 1: 41 48 83 86 17 | 83 86  6 31 17  9 48 53
    Card 2: 13 32 20 16 61 | 61 30 68 82 17 32 24 19
    Card 3:  1 21 53 59 44 | 69 82 63 72 16 21 14  1
    Card 4: 41 92 73 84 69 | 59 84 76 51 58  5 54 83
    Card 5: 87 83 26 28 32 | 88 30 70 12 93 22 82 36
    Card 6: 31 18 13 56 72 | 74 77 10 23 35 67 36 11
    """

In [65]:
test_case = [case.replace("    ", "") for case in test_case_copied.split("\n")[1:-1]]

In [66]:
test_case

['Card 1: 41 48 83 86 17 | 83 86  6 31 17  9 48 53',
 'Card 2: 13 32 20 16 61 | 61 30 68 82 17 32 24 19',
 'Card 3:  1 21 53 59 44 | 69 82 63 72 16 21 14  1',
 'Card 4: 41 92 73 84 69 | 59 84 76 51 58  5 54 83',
 'Card 5: 87 83 26 28 32 | 88 30 70 12 93 22 82 36',
 'Card 6: 31 18 13 56 72 | 74 77 10 23 35 67 36 11']

In [67]:
data = test_case

In [68]:
result_of_step_3_data = step_3(step_2(step_1(data)))

In [69]:
def step_a(result_of_step_3: dict[int, int]) -> dict[int, list[int]]:
    return {
        key: list(range(key + 1, key + value + 1)) # we do not actually need the list(), only for visual
        for key, value in result_of_step_3.items()
    }

In [70]:
result_of_step_a_data = step_a(result_of_step_3_data)
result_of_step_a_data

{1: [2, 3, 4, 5], 2: [3, 4], 3: [4, 5], 4: [5], 5: [], 6: []}

In [71]:
# Create a new dict, recursive_visit_dict, as a copy to result_of_step_a_data to hold all the states to be visited with recursed.
# initialise total_count=len(result_of_step_a_data) = 6, to count the original scratch card that we have for each.
# At key 1:
#    -  ONE INSTANCES OF SC1
#    We will have recursive_visit_dict[1]=[2,3,4,5].
#    There are four scratchcards at key 1, thus we add four to the count of total scratch cards: total_count = 6 + 4 = 10.
#    We now recurse: 
#    The first scratchcard that is won by scratchcard 1, is scratchcard 2.
#    append to recursive_visit_dict at key 2 the value of result_of_step_a_data[2].
#    The second scratchcard won by scratchcard 1 is scratchcard 3.
#    Thus append to recursive_visit_dict at key 3 the value of result_of_step_a_data[3]
#    The third scratchcard won by scratchcard 1 is scratchcard 4.
#    Thus append to recursive_visit_dict at key 4 the value of result_of_step_a_data[4]
#    The fourth scratchcard won by scratchcard 1 is scratchcard 5.
#    Thus append to recursive_visit_dict at key 5 the value of result_of_step_a_data[5]
# At key 2:
#    - TWO INSTANCES OF SC2
#    There will be four elements in recursive_visit_dict[2], The original [3, 4] and another copy of [3, 4] that came from key 1.
#    Thus add 4 to the count of the number of scratchcards. 6 + 4 + 4 = 14 in total now. 
#    Now we recurse: Run through recursive_visit_dict[2]=[3, 4, 3, 4].
#    The first element is 3, thus we append to recursive_visit_dict[3] the value of result_of_step_a_data[3]
#    The second element is 4, thus we append to recursive_visit_dict[4] the value of result_of_step_a_data[4]
#    The third element is 3, thus we append to recursive_visit_dict[3] the value of result_of_step_a_data[3]
#    The fourth element is 4, thus we append to recursive_visit_dict[4] the value of result_of_step_a_data[4]
# At key 3:
#    -- FOUR INSTANCES OF SC3
#    We will have recursive_visit_dict[3]=[4,5,4,5,4,5,4,5] as we have one original, another one that we got from key 1
#    one that we got from key 2, and 1 that we got from key 1 giving us another replica of key 2 that gives us one of key 3.
#    We thus add 8 to the total count of scratchcards giving us 6+4+4+8=22 until now.
#    Now we recurse: Run through recursive_visit_dict[3]=[4,5,4,5,4,5,4,5]
#    The first element is 4, thus we append to recursive_visit_dict[4] the value of result_of_step_a_data[4]
#    The second element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
#    The third element is 4, thus we append to recursive_visit_dict[4] the value of result_of_step_a_data[4]
#    The fourth element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
#    The fifth element is 4, thus we append to recursive_visit_dict[4] the value of result_of_step_a_data[4]
#    The sixth element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
#    The seventh element is 4, thus we append to recursive_visit_dict[4] the value of result_of_step_a_data[4]
#    The eight element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
# At key 4: 
#    --- EIGHT INSTANCES OF SC4
#    We will have eight elements of key 4: recursive_visit_dict[4] = [5,5,5,5,5,5,5,5]
#    We thus add 8 to the total_count=6+4+4+8+8=30
#    The first element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
#    The second element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
#    The third element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
#    The fourth element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
#    The fifth element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
#    The sixth element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
#    The seventh element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
#    The eight element is 5, thus we append to recursive_visit_dict[5] the value of result_of_step_a_data[5]
# At key 5:
#    --- FOURTEEN INSTANCES OF SC5
#    We will have zero elements in key 5: recursive_visit_dict[5]=[]
#    We thus add zero to the total_count =6+4+4+8+8+0=30
# At key 6:
#    --- ONE INSTANCE OF SC6
#    We will have zero elements in key 6: recursive_visit_dict[6]=[]
#    We thus add zero to the total_count=6+4+4+8+8+0+0=30

#### ??? possibly there are two counting systems here, that might each merit an implementation of its own. I am not entirely sure of the difference.

In [72]:
def step_b(result_of_step_a):
    recursive_visit_dict = copy.deepcopy(result_of_step_a)
    total_count = len(result_of_step_a)
    for outer_key in range(1, 1 + len(result_of_step_a)):
        print(f"outer_key={outer_key}, recursive_visit_dict[outer_key]={recursive_visit_dict[outer_key]}")
        total_count += len(recursive_visit_dict[outer_key])
        for inner_key in recursive_visit_dict[outer_key]:
            recursive_visit_dict[inner_key] = recursive_visit_dict[inner_key] + result_of_step_a[inner_key]
    return total_count

In [73]:
data = puzzle_input_split
data

['Card   1: 34 50 18 44 19 35 47 62 65 26 | 63  6 27 15 60  9 98  3 61 89 31 43 80 37 54 49 92 55  8  7 10 16 52 33 45',
 'Card   2: 90 12 98 56 22 99 73 46  1 28 | 52 77 32  8 81 41 53 22 28 46 48 27 98  1 94 12 99 72 84 90 92 73 24 63 56',
 'Card   3: 48 10 39 87 23 78 49 40 55  8 | 48 80 78 87 39 24 27 19 41 73 30 52 10  2 67 40 88 53 59 84 55 49  5 33 82',
 'Card   4: 21 45 91 26 64 51 42 84 11 94 | 55 56 36 65 84  2 68 44 52 58 86  6 33  7 97 40 30 14 39 80 82 57 79  1 10',
 'Card   5: 33  6 67 89 64 31 85 11  2 15 |  6 70 29 89 12 11 64 80  7 82 46 16 33 68 48 72 31  2 99 15 67 57  4 49 85',
 'Card   6: 51 20 11 66 38 39 69 48 25 74 | 39 74  3 86 19 25 21 55  2 38 46 60 66 82 51 11 98 88  8 48 49 94 20 69 72',
 'Card   7:  4 50 82 51 52 77 12 11 57 42 | 56 11 73 69 42 82 32 77 52 98 12 51 36 94 46  4 50 39 85 90 93 70 18 71 57',
 'Card   8: 96 31 27 93  7  8  6 23 15 72 | 55 79 86  4  6 35 12 27 95 29 73 81 87 43  7 13 62 15 72 71 58 48 63 94 89',
 'Card   9: 16 90 79 29 93 31 40

In [74]:
result_of_step_a = step_a(step_3(step_2(step_1(puzzle_input_split))))

In [75]:
step_b(result_of_step_a)

outer_key=1, recursive_visit_dict[outer_key]=[]
outer_key=2, recursive_visit_dict[outer_key]=[3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
outer_key=3, recursive_visit_dict[outer_key]=[4, 5, 6, 7, 8, 9, 10, 11, 4, 5, 6, 7, 8, 9, 10, 11]
outer_key=4, recursive_visit_dict[outer_key]=[5, 5, 5, 5]
outer_key=5, recursive_visit_dict[outer_key]=[6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
outer_key=6, recursive_visit_dict[outer_key]=[7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 7, 8

KeyboardInterrupt: 

##### ??? Does it make sense to parse over the dict in reserve, i.e., I guess, from the bottom of the tree? ??? 
##### !!! We want to transform the dictionary of which scratch cards each scratch card wins, to a dict of the number of scratch cards of that kind, original or copy, that have been had / the number of times a state has been visited. Strictly, we do not need the particular number for each scratchcard, only the sum of all the scratch cards. !!!

In [ ]:
def step_b(result_of_step_a: dict[int, list[int]]):
    # We take list of all copies that we win, and we need to
    # recurse on it to record the number of times each state is visited.
    # count = 0
    # for game in games:
    # 

    def inner(recursive):

    return {
        key: 
    }

### Checking runtime

#### Benchmarking

#### Runtime profiling